# Project interactive visualization
Using a dataset that your group is consider using for the term project, let's build some meaningful user-driven data visualization. Depending on your dataset this could include:
- Usage of geospatial information
- Building interactive views with widgets
- Organize multiple components into a simple dashboard

Keep these design principles in mind:
While you navigate through this notebook some things to take into consideration:
* Do not add interaction just to add it. Make sure it helps answer a question.
* Use meaningful titles and labels
* Document your code so it's readable and clean. If something does not work, document the issue and explain your best attempt.

In [ ]:
# ONLY INSTALL IF YOU HAVEN'T
!pip install jupyter_bokeh

In [ ]:
## These are most likely the libraries you will use
# Add or remove imports as needed

import pandas as pd
import numpy as np

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go

# Geospatial / geocoding
from geopy.geocoders import Nominatim

# Panel
import panel as pn
pn.extension('plotly')

In [ ]:
### Load your dataset here

from google.colab import drive
drive.mount('/content/drive')
path = '/content/drive/MyDrive/cs133/Cleaned'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import pandas as pd

# List files in the directory defined by 'path'
files_in_dir = os.listdir(path)
csv_files = [f for f in files_in_dir if f.endswith('.csv')]

# Dictionary to store DataFrames, named after their file names
dataframes_by_name = {}

for csv_file in csv_files:
    file_path = os.path.join(path, csv_file)
    df_name = os.path.splitext(csv_file)[0] # Get filename
    temp_df = pd.read_csv(file_path)
    dataframes_by_name[df_name] = temp_df
    print(f"Loaded DataFrame: {df_name}")

Loaded DataFrame: arrests_noncampus
Loaded DataFrame: arrests_on_campus
Loaded DataFrame: arrests_on_student_housing
Loaded DataFrame: arrests_public_property
Loaded DataFrame: criminal_offenses_noncampus
Loaded DataFrame: criminal_offenses_on_campus
Loaded DataFrame: criminal_offenses_on_student_housing
Loaded DataFrame: criminal_offenses_public_property
Loaded DataFrame: detailed_fire_data
Loaded DataFrame: disciplinary_actions_noncampus
Loaded DataFrame: disciplinary_actions_on_campus
Loaded DataFrame: disciplinary_actions_public_property
Loaded DataFrame: disciplinary_actions_student_housing
Loaded DataFrame: hate_crimes_noncampus
Loaded DataFrame: hate_crimes_on_campus
Loaded DataFrame: hate_crimes_student_housing
Loaded DataFrame: hate_crimes_public_property
Loaded DataFrame: total_fires
Loaded DataFrame: unfounded_crimes
Loaded DataFrame: vawa_offenses_noncampus
Loaded DataFrame: vawa_offenses_on_campus
Loaded DataFrame: vawa_offenses_student_housing
Loaded DataFrame: vawa_offen

## Question 1: Analytical Question
Write a question about your data that can be explored with an interactive plot. A good example would be a question where you can compare different categories. Explain how the plot help answer your question and why you choose that plot type.

Examples:
- Which regions have the highest average values?
- How does one variable compare across time?

In [ ]:

# Q1: How do 'Drug law violations' and 'Liquor law violations' arrests compare across different types of campus properties?
arrests_dfs = {
    'Non-Campus': dataframes_by_name['arrests_noncampus'],
    'On-Campus': dataframes_by_name['arrests_on_campus'],
    'Student Housing': dataframes_by_name['arrests_on_student_housing'],
    'Public Property': dataframes_by_name['arrests_public_property']
}

latest_year = max(df['Survey year'].max() for df in arrests_dfs.values())

plot_df = (pd.concat([df[df['Survey year'] == latest_year].assign(**{'Property Type': prop})
                      for prop, df in arrests_dfs.items()])
           .melt(id_vars='Property Type', value_vars=['Drug law violations', 'Liquor law violations'],
                 var_name='Violation Type', value_name='Count')
           .replace({'Drug law violations': 'Drug Law Violations', 'Liquor law violations': 'Liquor Law Violations'})
           .groupby(['Property Type', 'Violation Type'], as_index=False)['Count'].sum())

px.bar(plot_df, x='Property Type', y='Count', color='Violation Type', barmode='group',
       title=f'Drug vs. Liquor Law Violations by Campus Property Type ({latest_year})',
       labels={'Count': 'Number of Violations'}).show()

We picked a grouped bar chart because this type of chart makes it easy to see the difference between violation ttypes within each location and also the difference across locations.
On campus shows the highest drug violations as well as liquor violations, and all property types show relatively low liquor violations.

## Question 2. Create a simple interaction plot
Create a plot, it can be related to your question #1 or a new question, that users can interact with in some meaningful way. Explain in a markdown, how does the interaction add to the analysis.

Example of possible interactions:
*   Hover over information
*   Toogle between groups

In [ ]:
# Question 2: How have Violence Against Women Act (VAWA) offenses trended over time?

name_map = {'noncampus': 'Non-Campus', 'on_campus': 'On-Campus',
            'student_housing': 'Student Housing', 'public_property': 'Public Property'}

vawa_cols = ['Domestic violence', 'Dating violence', 'Stalking']

vawa_trends = (pd.concat([df.assign(**{'Property Type': next(v for k, v in name_map.items() if k in key)})
                           for key, df in dataframes_by_name.items() if key.startswith('vawa_offenses_')])
                 .groupby('Survey year', as_index=False)[vawa_cols].sum())

pivot = vawa_trends.set_index('Survey year')[vawa_cols].T

px.imshow(pivot,
          title='VAWA Offenses by Type Over Time',
          labels={'x': 'Year', 'y': 'Offense Type', 'color': 'Number of Offenses'},
          color_continuous_scale='Blues',
          aspect='auto').show()

The heatmap shows three VAWA offense types across every survey year at once across each campus property type (On-campus, non-campus, student housing, and public property). The hover interaction lets you see how many specific offenses have occured in the given year, making it easier to compare what offense type is more prevalent and rising/falling per year.

## Question 3. Choropleth Planning
Design a choropleth idea using your dataset.
In a markdown cell:
*  Identify the geographic unit you would map (state, county, country, ZIP code, etc.)
*  Identify the variable you would color by
*  Explain if any aggregation or preprocessing is needed
*  Briefly describe what GeoJSON file would be required

You do not need to have a perfect dataset for this question. However, your plan should be realistic.

If your data does not fully support a choropleth, build a prototype table that explains that structure you would need before mapping.

### Choropleth Idea: Drug and Liquor Law Violations in California Counties

**Geographic Unit:**
- California counties
- Reason: The dataset is heavily concentrated in California, so mapping at the state level would not show meaningful variation.
- County-level mapping allows for more detailed analysis and identification of local patterns.

**Variable to Color By:**
- Primary: Total Drug Law Violations
- Secondary (optional): Total Liquor Law Violations
- These variables represent counts and are appropriate for showing intensity across regions.

**Aggregation / Preprocessing Needed:**
1. Geographic Mapping:
   - Convert 'Institution name' or 'Unitid' into county-level data.
   - This may require external datasets (e.g., IPEDS) or geocoding.

2. Aggregation:
   - Group by:
        - County Name
        - Survey Year
   - Compute:
        - Sum of Drug Law Violations
        - Sum of Liquor Law Violations

3. Optional Normalization (recommended):
   - Normalize counts (e.g., per institution or per population)
   - Helps avoid bias toward larger counties.

**GeoJSON File Required:**
- California counties GeoJSON file
- Must include:
    - County boundaries
    - Matching key such as:
        - FIPS Code (preferred) or County Name

**Prototype Table Structure:**
| Survey Year | State | County        | FIPS Code | Drug Violations | Liquor Violations |
|-------------|-------|---------------|-----------|------------------|-------------------|
| 2015        | CA    | Santa Clara   | 06085     | 120              | 85                |
| 2015        | CA    | Los Angeles   | 06037     | 90               | 70                |
| 2016        | CA    | Santa Clara   | 06085     | 115              | 80                |
| 2016        | CA    | Los Angeles   | 06037     | 85               | 68                |

**Goal:**
- This choropleth would help identify geographic clusters and compare violation intensity across counties over time.


## Question 4. Geospatial Possibility Check

Determine whether your dataset can support a map-based visualization.

In a markdown cell, answer one of the following:
- If **yes**, explain what geographic fields you have and what type of map is appropriate.
- If **no**, explain what is missing and what you would need to create a map.

Write code that inspects or prepares the geographic column(s) you may use.

**Yes, this dataset can support a map-based visualization with minimal preprocessing.**

**Available Geographic Information:**
- 'Institution name' and 'Campus Name'
- All institutions are part of the California State University (CSU) system, meaning they are all located in California.

**Implication:**
- Since all schools are in California, we already know the **state-level geography**.
- This makes the dataset well-suited for a **California-focused map** rather than a national map.

**Limitation:**
- The dataset does NOT include:
    - County
    - Latitude/Longitude
    - FIPS codes

**What is Needed:**
- To create a more detailed map, we need to:
    1. Map each CSU campus to its **city or county**
    2. Optionally add:
        - Latitude/Longitude (for point maps)
        - County + FIPS codes (for choropleth maps)

**Appropriate Map Types:**
1. **Point Map (Best Option):**
   - Plot each CSU campus using latitude/longitude
   - Color or size points by:
        - Drug law violations
        - Liquor law violations

2. **Choropleth Map (Alternative):**
   - Aggregate violations by county
   - Requires mapping each campus to a county first

**Conclusion:**
- The dataset is well-suited for geospatial visualization within California.
- A point map of CSU campuses would be the most direct and effective approach.


In [ ]:
# Inspecting the 'Institution name' column, which is the closest to geographic information we have.
# This column needs further processing to derive mappable geographic data.

# Display unique institution names to get an idea of the data
# We'll just show the head to avoid a very long output.
print("Sample of Institution Names from combined_arrests_df:")
display(combined_arrests_df['Institution name'].head())

# Check for any obvious state names embedded directly (though unlikely to be consistent or complete)
# This is a basic check and not a substitute for proper geocoding.
print("\nExample values from 'Institution name' to show lack of direct state columns:")
for name in combined_arrests_df['Institution name'].sample(5):
    print(name)

# This demonstrates that a direct 'State' and/or 'County' column is missing.
# The next step would typically involve geocoding this information if we were to proceed.

Sample of Institution Names from combined_arrests_df:


,Institution name
0,California Polytechnic State University-San Lu...
1,California State Polytechnic University-Pomona
2,California State University Maritime Academy
3,California State University-Channel Islands
4,California State University-Chico



Example values from 'Institution name' to show lack of direct state columns:
San Jose State University
California State University-Northridge
California State University-Dominguez Hills
San Jose State University
San Jose State University


## Question 5. Geopy / Location Preparation

If your dataset has location names, addresses, cities, states, or countries, use geopy on a sample of your data to geocode locations or validate location information.

If your dataset does not have data that contains location, pick 5 destination you want to visit and use geopy to geocode locations.

In [ ]:
import time

# Initialize Nominatim geocoder
geolocator = Nominatim(user_agent="my_geocoder")

# Get unique institution names from the combined_arrests_df
unique_institutions = combined_arrests_df['Institution name'].unique()

# Create a dictionary to store geocoded results to avoid redundant API calls
geocoded_cache = {}

def geocode_institution(institution_name):
    if institution_name in geocoded_cache:
        return geocoded_cache[institution_name]

    # Add "California" to help Nominatim locate the institutions correctly
    try:
        location = geolocator.geocode(f"{institution_name}, California", timeout=10)
        if location:
            geocoded_info = {
                'latitude': location.latitude,
                'longitude': location.longitude,
                'address': location.address,
                'city': location.address.split(',')[0] if location.address else None
            }
        else:
            geocoded_info = {
                'latitude': None,
                'longitude': None,
                'address': None,
                'city': None
            }
    except Exception as e:
        print(f"Error geocoding {institution_name}: {e}")
        geocoded_info = {
            'latitude': None,
            'longitude': None,
            'address': None,
            'city': None
        }

    geocoded_cache[institution_name] = geocoded_info
    time.sleep(1) # Pause to respect Nominatim usage policy
    return geocoded_info

# Apply geocoding to unique institutions
geocoded_data = []
for inst_name in unique_institutions:
    geo_info = geocode_institution(inst_name)
    geocoded_data.append({'Institution name': inst_name, **geo_info})

# Create a DataFrame from geocoded results
geocoded_df = pd.DataFrame(geocoded_data)

# Merge the geocoded data back to the combined_arrests_df
# This will add latitude, longitude, and address info to each row based on institution name
combined_arrests_df_geo = pd.merge(
    combined_arrests_df,
    geocoded_df,
    on='Institution name',
    how='left'
)

print("Geocoding complete. Displaying first 5 rows of the new DataFrame with geographic data:")
display(combined_arrests_df_geo.head())

Geocoding complete. Displaying first 5 rows of the new DataFrame with geographic data:


,Survey year,Unitid,Institution name,OPEID,Campus ID,Campus Name,Institution Size,Illegal weapons possession,Drug law violations,Liquor law violations,Property Type,latitude,longitude,address,city
0,2015,110422,California Polytechnic State University-San Lu...,114300,1,Main Campus,20944,0,0,2,Non-Campus,35.174180,-120.741359,"California Polytechnic State University Pier, ...",California Polytechnic State University Pier
1,2015,110529,California State Polytechnic University-Pomona,114400,1,Main Campus,23717,0,0,0,Non-Campus,34.055669,-117.823928,"California State Polytechnic University, Pomon...",California State Polytechnic University
2,2015,111188,California State University Maritime Academy,113400,1,California State University Maritime Academy,1075,0,0,0,Non-Campus,NaN,NaN,None,None
3,2015,441937,California State University-Channel Islands,3980300,1,California State University Channel Islands - ...,6167,0,0,0,Non-Campus,34.167269,-119.045118,"California State University Channel Islands, 1...",California State University Channel Islands
4,2015,110538,California State University-Chico,114600,1,Main Campus,17220,1,2,0,Non-Campus,39.729501,-121.848110,"California State University, Chico, 400, West ...",California State University


## Question 6. Panel Widget

Create a Panel Widget that controls something in your analysis such as the ability to choose a column, category, year, etc.

The widget should affect an output such as a plot, table, or summary statistic.

In [ ]:
#combine all arrests DataFrames into one
all_arrests_data = []
for property_type, df in dataframes_by_name.items():
    if property_type.startswith('arrests_'):
        temp_df = df.copy()
        #standardize property type names
        if 'noncampus' in property_type:
            temp_df['Property Type'] = 'Non-Campus'
        elif 'on_campus' in property_type:
            temp_df['Property Type'] = 'On-Campus'
        elif 'on_student_housing' in property_type:
            temp_df['Property Type'] = 'Student Housing'
        elif 'public_property' in property_type:
            temp_df['Property Type'] = 'Public Property'
        else:
            temp_df['Property Type'] = property_type # Fallback
        all_arrests_data.append(temp_df)

combined_arrests_df = pd.concat(all_arrests_data, ignore_index=True)

#select relevant columns and melt for plotting
violations_melted_df = combined_arrests_df.melt(
    id_vars=['Survey year', 'Property Type'],
    value_vars=['Drug law violations', 'Liquor law violations'],
    var_name='Violation Type',
    value_name='Count'
)

#standardize Violation Type names
violations_melted_df['Violation Type'] = violations_melted_df['Violation Type'].replace({
    'Drug law violations': 'Drug Law Violations',
    'Liquor law violations': 'Liquor Law Violations'
})

#group by year, property type, and violation type to sum counts
trends_df = violations_melted_df.groupby(['Survey year', 'Property Type', 'Violation Type'], as_index=False)['Count'].sum()

#create a dropdown widget for Property Type
property_type_selector = pn.widgets.Select(
    name='Select Property Type',
    options=list(trends_df['Property Type'].unique())
)

#define a function to update the plot based on the selected property type
def update_plot(property_type):
    filtered_df = trends_df[trends_df['Property Type'] == property_type]
    fig = px.line(
        filtered_df,
        x='Survey year',
        y='Count',
        color='Violation Type',
        title=f'Drug vs. Liquor Law Violations Over Time on {property_type}',
        labels={'Survey year': 'Year', 'Count': 'Number of Violations'},
        hover_data={'Count': True, 'Survey year': True}
    )
    fig.update_layout(hovermode='x unified') # Improve hover experience
    return fig

#create a Panel dashboard with the widget and the plot
interactive_dashboard = pn.Column(
    "## Violation Trends by Campus Property Type",
    property_type_selector,
    pn.bind(update_plot, property_type=property_type_selector)
)

interactive_dashboard

Column
    [0] Markdown(str)
    [1] Select(options=['Non-Campus', ...], value='Non-Campus')
    [2] ParamFunction(function, _pane=Plotly, defer_load=False)

## Question 7. Mini Dashboard

1.   List item
2.   List item



Build a small Panel dashboard with at least two components. Arrange the components so that it is in a readable layout. Your dashboard should include:
* At least one plot,
* An additional element of your choice such as a widget, table, second plot, etc.

In [ ]:
melted_q7 = violations_melted_df.copy()
melted_q7['Count'] = pd.to_numeric(melted_q7['Count'], errors='coerce').fillna(0)

# widgets
min_year = int(melted_q7['Survey year'].min())
max_year = int(melted_q7['Survey year'].max())

year_slider = pn.widgets.RangeSlider(
    name='Survey Year Range',
    start=min_year,
    end=max_year,
    value=(min_year, max_year),
    step=1
)

violation_selector = pn.widgets.CheckBoxGroup(
    name='Violation Type',
    value=['Drug Law Violations', 'Liquor Law Violations'],
    options=['Drug Law Violations', 'Liquor Law Violations'],
    inline=True
)

In [ ]:
# bar chart
@pn.depends(year_slider, violation_selector)
def bar_chart(year_range, violation_types):
    if not violation_types:
        return pn.pane.Markdown("*Select at least one violation type above.*")

    filtered = melted_q7[
        (melted_q7['Survey year'] >= year_range[0]) &
        (melted_q7['Survey year'] <= year_range[1]) &
        (melted_q7['Violation Type'].isin(violation_types))
    ]
    grouped = filtered.groupby(['Property Type', 'Violation Type'], as_index=False)['Count'].sum()

    fig = px.bar(
        grouped,
        x='Property Type',
        y='Count',
        color='Violation Type',
        barmode='group',
        title=f'Total Violations by Property Type ({year_range[0]}–{year_range[1]})',
        labels={'Count': 'Total Violations', 'Property Type': 'Campus Property Type'},
        color_discrete_sequence=px.colors.qualitative.Set1
    )
    fig.update_layout(
        legend_title_text='Violation Type',
        hovermode='x unified',
        margin=dict(t=50, b=40)
    )
    return pn.pane.Plotly(fig, sizing_mode='stretch_width')

In [ ]:
# summary stats table
@pn.depends(year_slider, violation_selector)
def summary_table(year_range, violation_types):
    if not violation_types:
        return pn.pane.Markdown("*Select at least one violation type above.*")

    filtered = melted_q7[
        (melted_q7['Survey year'] >= year_range[0]) &
        (melted_q7['Survey year'] <= year_range[1]) &
        (melted_q7['Violation Type'].isin(violation_types))
    ]
    agg = (
        filtered.groupby('Property Type')['Count']
        .agg(Total='sum', Mean='mean', Max='max')
        .reset_index()
    )
    agg['Mean'] = agg['Mean'].round(1)
    agg = agg.sort_values('Total', ascending=False).reset_index(drop=True)

    return pn.widgets.DataFrame(
        agg,
        name='Summary Statistics by Property Type',
        sizing_mode='stretch_width',
        disabled=True
    )

In [ ]:
# dashboard
dashboard_q7 = pn.Column(
    pn.pane.Markdown("## CSU Campus Arrests Dashboard"),
    pn.Row(
        pn.Column("**Survey Year Range**", year_slider, width=350),
        pn.Column("**Violation Types**", violation_selector, width=300),
    ),
    pn.layout.Divider(),
    pn.pane.Markdown("### Total Violations by Property Type"),
    bar_chart,
    pn.layout.Divider(),
    pn.pane.Markdown("### Summary Statistics by Property Type"),
    summary_table,
    sizing_mode='stretch_width'
)

dashboard_q7

Column(sizing_mode='stretch_width')
    [0] Markdown(str)
    [1] Row
        [0] Column(width=350)
            [0] Markdown(str)
            [1] RangeSlider(end=2023, name='Survey Year Range', start=2015, step=1, value=(2015, 2023), value_end=2023, value_start=2015)
        [1] Column(width=300)
            [0] Markdown(str)
            [1] CheckBoxGroup(inline=True, name='Violation Type', options=['Drug Law Violations', ...], value=['Drug Law Violations', ...])
    [2] Divider()
    [3] Markdown(str)
    [4] ParamFunction(function, _pane=Plotly, defer_load=False)
    [5] Divider()
    [6] Markdown(str)
    [7] ParamFunction(function, _pane=DataFrame, defer_load=False)

## Question 8. Reflection

Write a short reflection addressing all of the following:
- Which interactive element was most useful in your notebook?
- What was the hardest part of working with your dataset?
- Did implementing interactive tools help your analysis? Why or why not?
- If you had more time, what would you improve or add?

The interactive element that was most useful was the combination of widgets + plots. The panel controls let me fitler by survey year range/property type/violation type. I think being able to dynamically adjust the view made it really easy to spot trends without having to create multiple static charts.


The hardest part of working with this dataset was handling fragmented info across many different CSV files. We had to standardize columns like Property Type across separate arrest tables and then reshape the data using melt and groupby in order to make the visualizations interactive.


Implementing interactive tools definitely helped my analysis because it made them easier to visualize. I could also look up multiple different patterns from one code block instead of having to create and analyze many different ones, which was really efficient.


If we had more time, it would be great if we could extend the geospatial part by finishing a point map of the CSU campuses and then linking it to the charts. For example, clicking a campus would update the time series and summary table.